# ch17 练习：性能与内存

与讲解版逐 Cell 对应，`assert` 验收保留。卡住回讲解版看对应小节。

## 一、考点回顾

| 竞赛评分点 | 本节覆盖 |
|---|---|
| 数据准备及处理 | category 降内存、query 高效筛选 |
| （隐性）工程能力 | 向量化替代循环 |

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

dev = pd.read_csv(DATA / "device_defects.csv")

print("pandas", pd.__version__, "| CoW 默认:", pd.options.mode.copy_on_write)
print("dev", dev.shape)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。"""
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


LOW_CARD = ["台区编号", "设备类型", "缺陷等级", "处理状态", "缺陷类型"]
KB = 1024

print("低基数列:", LOW_CARD)

## 题 1：CoW 行为验证（对应讲解 §3.1）

In [ ]:
#            把 sub 的 a 改成 99；原表 a 存入 after_sub
df = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
sub = df[df["a"] > 1]
sub["a"] = 99
after_sub = df["a"].tolist()

print("sub:", sub["a"].tolist(), "| 原表:", after_sub)
assert sub["a"].tolist() == [99, 99]
assert after_sub == [1, 2, 3], "CoW：切片修改不回写"

## 题 2：链式赋值现场（对应讲解 §3.2）

In [ ]:
#            把结果存入 after_chain；再用 loc 一步赋值修复
df2 = pd.DataFrame({"a": [1, 2, 3]})
df2[df2["a"] > 1]["a"] = 42
after_chain = df2["a"].tolist()
df2.loc[df2["a"] > 1, "a"] = 42

print("链式赋值后:", after_chain, "| loc 修复后:", df2["a"].tolist())
assert after_chain == [1, 2, 3], "链式赋值静默失效"
assert df2["a"].tolist() == [1, 42, 42]

## 题 3：内存体检 + category 压缩（对应讲解 §3.3-3.4）

In [ ]:
mu_obj = dev.memory_usage(deep=True)
mem_before = mu_obj.sum() / KB

#            deep 内存总和 KB 存入 mem_after
dev_cat = dev.copy()
dev_cat[LOW_CARD] = dev_cat[LOW_CARD].astype("category")
mem_after = dev_cat.memory_usage(deep=True).sum() / KB

print("内存:", mem_before, "KB →", mem_after, "KB")
print("设备类型 value_counts 一致:",
      bool(dev["设备类型"].value_counts().equals(dev_cat["设备类型"].value_counts())))
assert round(mem_before, 1) == 157.2
assert round(mem_after, 1) == 70.5
assert bool(dev["设备类型"].value_counts().equals(dev_cat["设备类型"].value_counts()))
assert dev_cat["设备类型"].cat.codes[:3].tolist() == [0, 2, 4]

## 题 4：query 等价性对账（对应讲解 §3.5）

In [ ]:
q = dev.query("负荷值 > 300 and 设备类型 == '变压器'")
mask = dev[(dev["负荷值"] > 300) & (dev["设备类型"] == "变压器")]

print("query:", len(q), "| mask:", len(mask), "| 索引一致:",
      bool(q.index.equals(mask.index)))
assert len(q) == 48 and len(mask) == 48
assert bool(q.index.equals(mask.index))
assert len(dev.query("负荷值 > 负荷值.mean() * 2")) == 7
assert len(dev.query("台区编号 in ['STATION_A_01', 'STATION_B_01']")) == 20

## 题 5：eval 造列（对应讲解 §3.6）

In [ ]:
#            NaN 个数存入 n_na
r = dev.eval("负荷比 = 负荷值 / 温度", inplace=False)
n_na = int(r["负荷比"].isna().sum())

print("r:", r.shape, "| 负荷比 NaN:", n_na)
assert r.shape == (270, 14)
assert n_na == 42

## 题 6（综合）：向量化改写并对账（对应讲解 §3.7）

In [ ]:
n = 200_000
big = pd.DataFrame({"x": np.random.default_rng(42).normal(100, 20, n)})
# （循环写法保留在挖空区外——它正是要被替代的"反面教材"）
t0 = time.perf_counter()
out_loop = np.empty(n)
for i, v in enumerate(big["x"].to_numpy()):
    out_loop[i] = (v - 100) / 20
t_loop = time.perf_counter() - t0
out_vec = (big["x"] - 100) / 20
same = np.allclose(out_loop, out_vec.to_numpy())

print(f"循环 {t_loop * 1000:.0f} ms")
print("结果一致:", bool(same))
assert bool(same), "向量化必须逐值一致"

---

## 综合自查

1. 题 2 里链式赋值为什么"不报错也不生效"？哪两种正确写法？
2. 题 3 压缩比约多少倍？哪些列**不能**这样转？
3. 题 4 的"等价"为什么必须对到索引级？
4. 题 7 循环里如果换成 `iterrows()` 会怎样？

全部答得上来，进入 ch18（可视化）。